In [14]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
llm = ChatOpenAI(

    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ["GITHUB_TOKEN"],
    model="gpt-4o",
    temperature=0.7,
    streaming=True,
    max_tokens=4096,
    top_p=1
)

sesion_unimarc = {}

In [ ]:
def historial_de_conversacion(sesion_id : str):
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]

def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        mensajes_a_resumir = historial.messages[:-2]

        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        summary_response = llm.invoke(f"Resume esta conversación de inventario en 2 líneas:\n{conversation_text}")
        summary = summary_response.content
        
        recent_messages = historial.messages[-2:]
        historial.clear()
        historial.add_ai_message(f"[RESUMEN]: {summary}")
        historial.messages.extend(recent_messages)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de Inventario de Unimarc. Ayudas a los usuarios a gestionar su inventario, responder preguntas sobre productos, y proporcionar información relevante sobre el stock y las operaciones de la tienda. Tienes que ser amigable, eficiente y siempre estar dispuesto a ayudar. Si no sabes la respuesta a una pregunta, es mejor admitirlo que inventar una respuesta incorrecta. Siempre debes mantener un tono profesional y cortés en tus respuestas. Las respuestas deben ser breves y al punto, evitando información innecesaria. Si el usuario hace una pregunta que no está relacionada con el inventario o las operaciones de la tienda, debes redirigir la conversación de vuelta a temas relevantes para Unimarc. Recuerda que tu objetivo principal es ayudar a los usuarios a gestionar su inventario de manera efectiva y proporcionar información precisa sobre los productos y operaciones de la tienda."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejecutar_chat(input_text, session_id):
    sincronizar_contexto_stock(session_id)
    
    config = {"configurable": {"session_id": session_id}}
    
    print(f"[OUTPUT]: ", end="", flush=True)
    
    try:
        for chunk in conversation.stream({"input": input_text}, config=config):
            print(chunk.content, end="", flush=True)
        print("\n")
    except Exception as e:
        print(f"\n[ERROR_LOG]: {e}")

id_actual = "SYS-LOG-01"
print(f"TERMINAL DE GESTIÓN UNIMARC | SESIÓN: {id_actual}")
print("-" * 50)

while True:
    user_input = input("[INPUT]: ")
    
    if user_input.lower() in ["exit", "quit", "salir"]:
        print("[SISTEMA]: Sesión finalizada.")
        break
    
    if user_input.strip():
        ejecutar_chat(user_input, id_actual)

TERMINAL DE GESTIÓN UNIMARC | SESIÓN: SYS-LOG-01
--------------------------------------------------
[OUTPUT]: ¡Hola! Encantado de ayudarte con la gestión de inventario en tu sucursal de Concepción. Por favor, dime en qué necesitas apoyo o qué información buscas.

[OUTPUT]: Entendido. Para resolver este descuadre en el stock de detergentes, te sugiero seguir estos pasos:

1. **Verificar movimientos recientes:** Revisa las entradas y salidas en el sistema para detectar posibles errores en el registro.
2. **Confirmar ventas o promociones:** Asegúrate de que no haya habido ventas no ingresadas o promociones que expliquen la diferencia.
3. **Revisar posibles mermas:** Confirma si hubo productos dañados o vencidos que no se hayan registrado como merma.
4. **Auditar físicamente:** Realiza un recuento físico detallado del área de almacenamiento para confirmar las unidades disponibles.
5. **Reportar el descuadre:** Si no se encuentra el motivo, informa el incidente al área de auditoría o al enc

Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.smith.langchain.com', port=443) at 0x194fea7a5d0>, 'Connection to api.smith.langchain.com timed out. (connect timeout=3)'))"))
Content-Length: 16242
API Key: lsv2_********************************************72trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-2bc9-7593-a173-6d1e11743ab8; trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-2bff-7382-9bfd-3db824e2f383; trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-2c19-7a11-8dec-8b4207a01407; trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-2c1e-78c3-a096-c4119badb1fa; trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-2c22-7ba1-a033-b88e64e8

[OUTPUT]: Lo siento, no tengo información sobre eventos deportivos. ¿Puedo ayudarte con algo relacionado con el inventario o las operaciones de la tienda? 😊

[SISTEMA]: Sesión finalizada.


Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.smith.langchain.com', port=443) at 0x194fea79bd0>, 'Connection to api.smith.langchain.com timed out. (connect timeout=3)'))"))
Content-Length: 90948
API Key: lsv2_********************************************72trace=019d72d0-2bc9-7593-a173-6d1e11743ab8,id=019d72d0-3ad2-7e70-bf4e-ad2b26238b66; trace=019d72d0-74c0-7151-9d86-495c594fad5a,id=019d72d0-74c0-7151-9d86-495c594fad5a; trace=019d72d0-74c0-7151-9d86-495c594fad5a,id=019d72d0-74c2-75e3-bb8b-7d40f8267711; trace=019d72d0-74c0-7151-9d86-495c594fad5a,id=019d72d0-74c6-7441-b3b1-405f59032a2b; trace=019d72d0-74c0-7151-9d86-495c594fad5a,id=019d72d0-74cc-7c53-b474-9cf4dbe6